
# Card &middot; Ensembles

| | |
|---|---|
| **time** | ~20 minutes |
| **GPU** | not needed |
| **typical gain** | small but nearly free |
| **needs** | at least two saved prediction files |

The cheapest improvement available to you. Three of the top ten finishers in
the real challenge combined a message-passing neural network with classical
gradient boosting, and the fourth-place entry was explicitly a hybrid.

The reason it works: two models that are wrong in *different* ways partially
cancel. Which means the useful question is not "which of my models is best"
but "how different are my models from each other".

In [ ]:
# Run me first.
%pip -q install rdkit pandas numpy scipy scikit-learn huggingface_hub fsspec matplotlib seaborn

# Get common.py. If you uploaded it yourself (folder icon in the left sidebar),
# this leaves your copy alone -- it only downloads when the file is missing.
!test -s common.py || wget -q -O common.py https://raw.githubusercontent.com/CHANGE-ME/admet-hackathon/main/common.py

import os, sys
assert os.path.exists("common.py") and os.path.getsize("common.py") > 1000, (
    "common.py is missing or truncated. Upload it using the folder icon in the "
    "left sidebar, then re-run this cell.")

sys.modules.pop("common", None)   # force a fresh read if you just re-uploaded it
import common
common.setup(pair="CHANGE-ME")

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_style("whitegrid")

train = common.load_train()
test  = common.load_test()
common.list_predictions()

If that table is empty, run `02_validation` (which saves a
LightGBM baseline) or any card first. You need at least two.

In [ ]:
NAMES = common.list_predictions()["name"].tolist()
models = {n: common.load_predictions(n) for n in NAMES}
print(f"loaded {len(models)}:", ", ".join(models))

---
## 1. How different are your models?

Correlate the models' predictions with each other. **Low correlation is what
you want.** Two models at 0.99 correlation will ensemble to almost exactly
themselves; two at 0.85 have genuinely different information.

In [ ]:
ENDPOINT = "LogD"
mat = pd.DataFrame({n: m.set_index(common.ID_COL)[ENDPOINT] for n, m in models.items()})
corr = mat.corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="RdBu_r", vmin=0.7, vmax=1, ax=ax)
ax.set_title(f"Agreement between your models ({ENDPOINT})")
plt.tight_layout(); plt.show()

### &#9654;&#65039; Predict first

**Averaging your two most-similar models, versus averaging your two least-similar models: which will score better?**

Write your answer here before running the next cell &mdash; one line is enough:

> `your prediction:`

---
## 2. Three ways to combine

In [ ]:
def average(names, weights=None):
    """Plain (or weighted) mean of predictions."""
    frames = [models[n].set_index(common.ID_COL)[common.ENDPOINTS] for n in names]
    w = np.ones(len(frames)) if weights is None else np.asarray(weights, float)
    w = w / w.sum()
    stacked = sum(f * wi for f, wi in zip(frames, w))
    return stacked.reset_index()


def rank_average(names):
    """Average the RANKS, not the values.

    Use this when your models are on slightly different scales, or when you
    care about ordering compounds rather than absolute numbers -- which, for a
    medicinal chemist choosing what to make next, is usually the case.
    """
    frames = [models[n].set_index(common.ID_COL)[common.ENDPOINTS].rank(pct=True)
              for n in names]
    return (sum(frames) / len(frames)).reset_index()


def per_endpoint_best(names, choice):
    """Different model per endpoint. choice maps endpoint -> model name."""
    out = models[names[0]].set_index(common.ID_COL)[common.ENDPOINTS].copy()
    for e, n in choice.items():
        out[e] = models[n].set_index(common.ID_COL)[e]
    return out.reset_index()

A caution on `rank_average`: it destroys the absolute scale, so
MAE and RAE will look terrible even if the ordering improves. Only reach for it
if you have decided that ranking is what you care about &mdash; and if you do,
say so on slide 3.

---
## 3. Does it actually help?

You cannot score against the test set, so evaluate the ensemble the same way
you evaluated everything else: on your own validation fold. That means
re-running your models under the split, which the cards already did &mdash; so
here we compare on held-out training molecules.

In [ ]:
fold, _ = common.load_split(train)
va = (fold == "val").to_numpy()
truth = train.loc[va].reset_index(drop=True)
val_ids = set(truth[common.ID_COL])

usable = {n: m for n, m in models.items()
          if len(val_ids & set(m[common.ID_COL])) > 50}
if not usable:
    print("Your saved predictions cover the TEST molecules only, so they cannot be\n"
          "scored against held-out training data. Combine them anyway (below) and\n"
          "judge the ensemble by whether the members were individually sound.")
else:
    for n, m in usable.items():
        print(f"{n:28s} {common.evaluate(truth, m)['RAE'].mean():.3f}")

---
## 4. Build and save your ensemble

In [ ]:
MEMBERS = NAMES[:2]          # <-- choose deliberately, not just the top two
WEIGHTS = None               # e.g. [0.6, 0.4]

ens = average(MEMBERS, WEIGHTS)
ens = ens[ens[common.ID_COL].isin(test[common.ID_COL])]
common.save_predictions(ens, "ensemble",
                        note=f"mean of {MEMBERS}")

### If you have time

- Weight members by their individual validation score instead of equally.
- Use a *different* ensemble per endpoint &mdash; your fingerprint model may win
  on efflux while your descriptor model wins on LogD. `per_endpoint_best` does
  this. Watch out: choosing per-endpoint winners on your validation fold is
  itself a form of fitting, and with nine endpoints and a handful of models you
  can overfit that choice.
- Snapshot ensembling and the Caruana greedy selection method both appeared at
  the top of the real leaderboard. See `FRONTIER.md`.